In [2]:
import pandas as pd
import numpy as np

# ============================================================
# Load malicious random 10% dataset
# ============================================================

det_path = "uav_detection_random_10.csv"
rl_path = "uav_rl_random_10.csv"

det_df = pd.read_csv(det_path)
rl_df = pd.read_csv(rl_path)

print("===== DATASET SHAPES =====")
print("Detection shape:", det_df.shape)
print("RL shape:", rl_df.shape)

print("\n===== DETECTION COLUMNS =====")
print(det_df.columns.tolist())

print("\n===== RL COLUMNS =====")
print(rl_df.columns.tolist())

# ============================================================
# Expected row checks
# ============================================================

expected_detection_rows = 4 * 3 * 120 * 20 * 19
expected_rl_rows = 4 * 3 * 120 * 20

print("\n===== EXPECTED ROW CHECK =====")
print("Expected detection rows:", expected_detection_rows)
print("Actual detection rows:", len(det_df))
print("Detection row check:", "PASS" if len(det_df) == expected_detection_rows else "FAIL")

print("\nExpected RL rows:", expected_rl_rows)
print("Actual RL rows:", len(rl_df))
print("RL row check:", "PASS" if len(rl_df) == expected_rl_rows else "FAIL")

# ============================================================
# Label / flag checks
# ============================================================

print("\n===== LABEL COUNTS =====")
print("actual_target_malicious:")
print(det_df["actual_target_malicious"].value_counts().sort_index())

print("\nreceiver_malicious:")
print(det_df["receiver_malicious"].value_counts().sort_index())

print("\nphase1_fail:")
print(det_df["phase1_fail"].value_counts().sort_index())

print("\nphase2_fail:")
print(det_df["phase2_fail"].value_counts().sort_index())

print("\nfinal_flag:")
print(det_df["final_flag"].value_counts().sort_index())

# ============================================================
# NaN / Inf checks
# ============================================================

numeric_det = det_df.select_dtypes(include=[np.number])
numeric_rl = rl_df.select_dtypes(include=[np.number])

print("\n===== NaN CHECK =====")
print("Detection total NaN:", det_df.isna().sum().sum())
print("RL total NaN:", rl_df.isna().sum().sum())

print("\n===== INF CHECK =====")
print("Detection total Inf:", np.isinf(numeric_det).sum().sum())
print("RL total Inf:", np.isinf(numeric_rl).sum().sum())

print("\n===== ENVIRONMENT COUNTS =====")
print(det_df["envId"].value_counts().sort_index())

print("\n===== FORMATION COUNTS =====")
print(det_df["formationType"].value_counts().sort_index())

===== DATASET SHAPES =====
Detection shape: (547200, 19)
RL shape: (28800, 14)

===== DETECTION COLUMNS =====
['envId', 'formationType', 'timeStep', 'receiverId', 'targetId', 'd_true', 'd_gps', 'd_rssi', 'delta_d', 'phase1_fail', 'jury1', 'jury2', 'jury3', 'maxResidual', 'phase2_fail', 'final_flag', 'actual_target_malicious', 'receiver_malicious', 'malicious_ratio']

===== RL COLUMNS =====
['envId', 'formationType', 'timeStep', 'receiverId', 'sigma_rssi_bar', 'sigma_gps_bar', 'snr_bar', 'plr_bar', 'relative_speed_bar', 'relative_height_bar', 'trusted_count', 'trust_bar', 'theta', 'malicious_ratio']

===== EXPECTED ROW CHECK =====
Expected detection rows: 547200
Actual detection rows: 547200
Detection row check: PASS

Expected RL rows: 28800
Actual RL rows: 28800
RL row check: PASS

===== LABEL COUNTS =====
actual_target_malicious:
actual_target_malicious
0    492480
1     54720
Name: count, dtype: int64

receiver_malicious:
receiver_malicious
0    492480
1     54720
Name: count, dtype:

In [3]:
import numpy as np
import pandas as pd

# ============================================================
# Safe metric function
# ============================================================

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))

    total = TN + FP + FN + TP

    accuracy = (TP + TN) / total if total > 0 else np.nan
    precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
    recall = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if not np.isnan(precision) and not np.isnan(recall) and (precision + recall) > 0
        else np.nan
    )
    fp_rate = FP / (FP + TN) if (FP + TN) > 0 else np.nan
    fn_rate = FN / (FN + TP) if (FN + TP) > 0 else np.nan

    return {
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fp_rate": fp_rate,
        "fn_rate": fn_rate,
    }


def print_metrics(title, metrics):
    print(f"\n===== {title} =====")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


# ============================================================
# Overall metrics
# ============================================================

y_true = det_df["actual_target_malicious"].astype(int)

phase1_metrics = compute_metrics(y_true, det_df["phase1_fail"].astype(int))
phase2_metrics = compute_metrics(y_true, det_df["phase2_fail"].astype(int))
final_metrics = compute_metrics(y_true, det_df["final_flag"].astype(int))

print_metrics("PHASE 1 ONLY METRICS", phase1_metrics)
print_metrics("PHASE 2 ONLY METRICS", phase2_metrics)
print_metrics("FINAL DETECTION METRICS", final_metrics)

# Save
overall_metrics_df = pd.DataFrame([
    {"method": "phase1_only", **phase1_metrics},
    {"method": "phase2_only", **phase2_metrics},
    {"method": "final_phase1_or_phase2", **final_metrics},
])

overall_metrics_df.to_csv("random_10_overall_metrics.csv", index=False)

print("\nSaved:")
print("random_10_overall_metrics.csv")


===== PHASE 1 ONLY METRICS =====
TN: 351927
FP: 140553
FN: 8526
TP: 46194
accuracy: 0.7276
precision: 0.2474
recall: 0.8442
f1: 0.3826
fp_rate: 0.2854
fn_rate: 0.1558

===== PHASE 2 ONLY METRICS =====
TN: 229507
FP: 262973
FN: 46204
TP: 8516
accuracy: 0.4350
precision: 0.0314
recall: 0.1556
f1: 0.0522
fp_rate: 0.5340
fn_rate: 0.8444

===== FINAL DETECTION METRICS =====
TN: 88954
FP: 403526
FN: 10
TP: 54710
accuracy: 0.2625
precision: 0.1194
recall: 0.9998
f1: 0.2133
fp_rate: 0.8194
fn_rate: 0.0002

Saved:
random_10_overall_metrics.csv


In [4]:
# ============================================================
# Phase contribution analysis
# ============================================================

malicious_df = det_df[det_df["actual_target_malicious"] == 1].copy()
benign_df = det_df[det_df["actual_target_malicious"] == 0].copy()

total_malicious = len(malicious_df)
total_benign = len(benign_df)

mal_caught_phase1 = int((malicious_df["phase1_fail"] == 1).sum())
mal_passed_phase1 = int((malicious_df["phase1_fail"] == 0).sum())

mal_caught_phase2 = int((malicious_df["phase2_fail"] == 1).sum())
mal_final_caught = int((malicious_df["final_flag"] == 1).sum())

mal_pass_phase1_df = malicious_df[malicious_df["phase1_fail"] == 0].copy()

if len(mal_pass_phase1_df) > 0:
    phase2_catch_after_phase1_pass = mal_pass_phase1_df["phase2_fail"].mean()
else:
    phase2_catch_after_phase1_pass = np.nan

benign_flagged_phase1 = int((benign_df["phase1_fail"] == 1).sum())
benign_flagged_phase2 = int((benign_df["phase2_fail"] == 1).sum())
benign_final_flagged = int((benign_df["final_flag"] == 1).sum())

print("===== PHASE CONTRIBUTION CHECK =====")

print("\n--- Malicious rows ---")
print("Total malicious rows:", total_malicious)
print("Malicious caught by Phase 1:", mal_caught_phase1)
print("Malicious passed Phase 1:", mal_passed_phase1)
print("Malicious caught by Phase 2:", mal_caught_phase2)
print("Malicious finally caught:", mal_final_caught)

print("\nPhase 1 malicious recall:")
print(mal_caught_phase1 / total_malicious if total_malicious > 0 else np.nan)

print("\nFinal malicious recall:")
print(mal_final_caught / total_malicious if total_malicious > 0 else np.nan)

print("\nPhase 2 catch rate among malicious that passed Phase 1:")
print(phase2_catch_after_phase1_pass)

print("\n--- Benign rows ---")
print("Total benign rows:", total_benign)
print("Benign flagged by Phase 1:", benign_flagged_phase1)
print("Benign flagged by Phase 2:", benign_flagged_phase2)
print("Benign finally flagged:", benign_final_flagged)

print("\nPhase 1 FP rate:")
print(benign_flagged_phase1 / total_benign if total_benign > 0 else np.nan)

print("\nFinal FP rate:")
print(benign_final_flagged / total_benign if total_benign > 0 else np.nan)

phase_contribution = pd.DataFrame([{
    "attack_type": "random",
    "malicious_ratio": 0.10,
    "total_malicious": total_malicious,
    "mal_caught_phase1": mal_caught_phase1,
    "mal_passed_phase1": mal_passed_phase1,
    "mal_caught_phase2": mal_caught_phase2,
    "mal_final_caught": mal_final_caught,
    "phase1_malicious_recall": mal_caught_phase1 / total_malicious if total_malicious > 0 else np.nan,
    "final_malicious_recall": mal_final_caught / total_malicious if total_malicious > 0 else np.nan,
    "phase2_catch_after_phase1_pass": phase2_catch_after_phase1_pass,
    "phase1_fp_rate": benign_flagged_phase1 / total_benign if total_benign > 0 else np.nan,
    "final_fp_rate": benign_final_flagged / total_benign if total_benign > 0 else np.nan,
}])

phase_contribution.to_csv("random_10_phase_contribution.csv", index=False)

print("\nSaved:")
print("random_10_phase_contribution.csv")

===== PHASE CONTRIBUTION CHECK =====

--- Malicious rows ---
Total malicious rows: 54720
Malicious caught by Phase 1: 46194
Malicious passed Phase 1: 8526
Malicious caught by Phase 2: 8516
Malicious finally caught: 54710

Phase 1 malicious recall:
0.844188596491228

Final malicious recall:
0.9998172514619883

Phase 2 catch rate among malicious that passed Phase 1:
0.9988271170537181

--- Benign rows ---
Total benign rows: 492480
Benign flagged by Phase 1: 140553
Benign flagged by Phase 2: 262973
Benign finally flagged: 403526

Phase 1 FP rate:
0.28539839181286547

Final FP rate:
0.8193754061078623

Saved:
random_10_phase_contribution.csv


In [5]:
# ============================================================
# Random 10% attack: metrics by environment
# ============================================================

env_labels = {
    0: "Perfect",
    1: "Open",
    2: "Suburban",
    3: "DenseUrban"
}

rows = []

for env_id, sub in det_df.groupby("envId"):
    y_env = sub["actual_target_malicious"].astype(int)

    m_phase1 = compute_metrics(y_env, sub["phase1_fail"].astype(int))
    m_final = compute_metrics(y_env, sub["final_flag"].astype(int))

    rows.append({
        "envId": env_id,
        "envName": env_labels.get(env_id, str(env_id)),

        "phase1_recall": m_phase1["recall"],
        "phase1_fp_rate": m_phase1["fp_rate"],
        "phase1_precision": m_phase1["precision"],
        "phase1_f1": m_phase1["f1"],

        "final_recall": m_final["recall"],
        "final_fp_rate": m_final["fp_rate"],
        "final_precision": m_final["precision"],
        "final_f1": m_final["f1"],

        "final_TN": m_final["TN"],
        "final_FP": m_final["FP"],
        "final_FN": m_final["FN"],
        "final_TP": m_final["TP"],
    })

env_summary = pd.DataFrame(rows)

print("===== RANDOM 10% METRICS BY ENVIRONMENT =====")
print(env_summary)

env_summary.to_csv("random_10_metrics_by_environment.csv", index=False)

print("\nSaved:")
print("random_10_metrics_by_environment.csv")

===== RANDOM 10% METRICS BY ENVIRONMENT =====
   envId     envName  phase1_recall  phase1_fp_rate  phase1_precision  \
0      0     Perfect       0.821637        0.086111          0.514605   
1      1        Open       0.839181        0.162094          0.365175   
2      2    Suburban       0.852339        0.335997          0.219884   
3      3  DenseUrban       0.863596        0.557391          0.146867   

   phase1_f1  final_recall  final_fp_rate  final_precision  final_f1  \
0   0.632847      0.999708       0.302883         0.268330  0.423098   
1   0.508899      0.999708       0.978826         0.101916  0.184975   
2   0.349583      0.999854       0.996142         0.100335  0.182370   
3   0.251041      1.000000       0.999651         0.100031  0.181870   

   final_TN  final_FP  final_FN  final_TP  
0     85829     37291         4     13676  
1      2607    120513         4     13676  
2       475    122645         2     13678  
3        43    123077         0     13680  

Saved: